# LC 560 — Subarray Sum Equals K
**Difficulty:** Medium | **Category:** Array / HashMap | **Pattern:** Prefix Sum + Complement

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Track a running prefix sum as you
scan. If <code>prefix_sum - k</code> has appeared before, then the
subarray between that earlier point and now sums to exactly k.
Count how many times that complement appeared.
</div>


## Official Problem Statement

Given an array of integers `nums` and an integer `k`, return
the **total number of subarrays** whose sum equals to `k`.

A **subarray** is a contiguous **non-empty** sequence of elements
within an array.

**Constraints:**
- `1 <= nums.length <= 2 * 10^4`
- `-1000 <= nums[i] <= 1000`
- `-10^7 <= k <= 10^7`


## What This Is Actually Asking

You have a list of numbers (can be negative) and a target sum k.
Count every contiguous slice of the list whose numbers add up
to k.
Slices can overlap and can be any length.
Because values can be negative, a sliding window won't work —
you need prefix sums.


## Walk Through an Example by Hand

```
nums = [1, 1, 1], k = 2

Build prefix sums:
  index:  0   1   2
  nums:   1   1   1
  prefix: 1   2   3

Scan with map = {0: 1}  (0 seen once before any element)

 i | num | prefix_sum | need (ps-k) | map before   | count
---+-----+------------+-------------+--------------+------
 0 |  1  |     1      |    1-2=-1   | {0:1}        |  0
   |     |            |   not found | store {0:1,1:1}
 1 |  1  |     2      |    2-2= 0   | {0:1,1:1}    |  1  ← map[0]=1
   |     |            |   found! +1 | store {0:1,1:1,2:1}
 2 |  1  |     3      |    3-2= 1   | {...,2:1}    |  2  ← map[1]=1
   |     |            |   found! +1 | store {..,3:1}

Answer: 2
```


## The Picture

```
nums    =  [ 1,  1,  1 ]   k = 2
prefix  =  [ 1,  2,  3 ]   (running total)

Subtraction trick:

  prefix[j] - prefix[i] = sum of subarray (i+1 .. j)

  We want that sum == k:
    prefix[j] - prefix[i] == k
    prefix[i] == prefix[j] - k

So: for each j, ask "how many earlier prefix sums equal
    (current_prefix - k)?"  That count is stored in our map.

HashMap: { prefix_sum → how many times seen }

 After i=0: {0:1, 1:1}
 After i=1: {0:1, 1:1, 2:1}   count so far: 1
 After i=2: {0:1, 1:1, 2:1, 3:1}  count so far: 2

Seed with {0: 1} to handle subarrays starting at index 0.
```


## When To Use This Pattern

- When the array has **negative numbers** and you need subarray
  sums, think prefix sum (sliding window won't work).
- When you need to **count** subarrays with a target sum,
  think prefix sum + HashMap.
- When you see `sum(i..j) == k`, think
  `prefix[j] - prefix[i] == k` → complement lookup.
- When you must count overlapping sub-problems efficiently,
  think frequency map of prefix sums.
- Seed the map with `{0: 1}` whenever the subarray could start
  at index 0.


## The Approach

Maintain a running `prefix_sum` as you iterate.
Keep a HashMap that records how many times each prefix sum
value has been seen; initialise it with `{0: 1}` to handle
subarrays that start at the beginning.
At each position, add `map[prefix_sum - k]` to your count
(the number of earlier positions where the complement appeared),
then increment the map entry for the current prefix sum.
Return the total count after one pass.


In [ ]:
from typing import List                   # typed signatures
from collections import defaultdict       # auto-initialise int counter


In [ ]:
def test_harness(func):
    """Run test cases for Subarray Sum Equals K."""
    tests = [
        # (nums, k, expected_count)
        ([1, 1, 1],          2, 2),
        ([1, 2, 3],          3, 2),
        ([1],                1, 1),
        ([1],                0, 0),
        ([-1, -1, 1],        0, 1),
        ([0, 0, 0],          0, 6),  # all sub-arrays sum to 0
        ([1, -1, 1, -1, 1], 0, 4),
    ]
    passed = 0
    for nums, k, expected in tests:
        result = func(nums, k)
        if result == expected:
            print(f"PASSED | nums={nums}, k={k} → {result}")
            passed += 1
        else:
            print(
                f"FAILED | nums={nums}, k={k} "
                f"| got {result}, expected {expected}"
            )
    print(f"\n{passed}/{len(tests)} tests passed.")


In [ ]:
def subarray_sum(nums: List[int], k: int) -> int:
    """
    Count contiguous subarrays whose elements sum to k.

    Approach:
      Running prefix_sum + HashMap of frequencies.
      Seed map with {0:1}. At each index add map[prefix_sum - k]
      to count, then store current prefix_sum in map.
      If prefix_sum - k was seen N times before, N subarrays
      ending here sum to k.

    Time:  O(n)  — single pass
    Space: O(n)  — HashMap stores up to n distinct prefix sums
    """
    pass


# --- debug prints (expected in comments) ---
print(subarray_sum([1, 1, 1], 2))           # 2
print(subarray_sum([1, 2, 3], 3))           # 2
print(subarray_sum([1], 1))                 # 1
print(subarray_sum([-1, -1, 1], 0))         # 1
print(subarray_sum([0, 0, 0], 0))           # 6


In [ ]:
# Uncomment and run when solution is ready
# test_harness(subarray_sum)


## Complexity

| Approach                  | Time   | Space  |
|---------------------------|--------|--------|
| Brute Force (nested loops)| O(n²)  | O(1)   |
| Prefix Sum + HashMap      | O(n)   | O(n)   |


## Real World Connection

At Citi, monitoring 6,000 endpoints generates a continuous
stream of metric deltas; finding any window of time where the
cumulative error count crosses a threshold is exactly the
subarray-sum-equals-K problem applied to a time series.
An AWS Lambda processing DynamoDB streams can use a prefix sum
over byte counts to detect when a batch window has accumulated
exactly the target payload size before flushing.
ETL pipelines that must identify consecutive records whose
combined value matches a reconciliation target apply this same
one-pass prefix-sum HashMap to avoid O(n²) nested scans across
millions of rows.


> **Simplicity and clarity is Gold.** — Sean's Study Mantra
